In [2]:
import numpy as np 
from numpy.linalg import inv

<table><tr>
<td> <img src="data/ek1.png" alt="EKF short example 1" width="600"/> </td>
<td> <img src="data/ek3.png" alt="EKF short example 1" width="600"/> </td>
</tr>

<tr>
<td> <img src="data/ekf.png" alt="EKF" width="600"/> </td>
</tr>
</table>


In [ ]:
dt = 0.5
u0 = -2
z1 = np.pi/6
R = 0.1 * np.eye(2)
Q = 0.01 * np.eye(1)

def predict(t, u, dt, R):
    prev_mu, prev_sigma = belief(t-1)
    mu = g(u, prev_mu, dt)
    G = partial_g_partial_x(dt) 
    sigma = G @ prev_sigma @ G.T + R
    return mu, sigma

def h(x, S, D):
    p = x[0, 0]
    return np.arctan2(S, (D-p))

def g(u, prev_x, dt):
    Gx = partial_g_partial_x(dt)
    Gu = partial_g_partial_u(dt)
    return Gx @ prev_x + Gu * u 

def partial_g_partial_x(dt):
    Gx = np.eye(2)
    Gx[0, 1] = dt
    return Gx

def partial_g_partial_u(dt):
    Gu = np.zeros([2, 1])
    Gu[1, 0] = dt
    return Gu

def partial_h_partial_x(x, dt, S, D):
    p = x[0, 0]
    H = np.zeros([1, 2])
    H[0, 0] = S / ( (D-p)**2 + S**2 )
    return H

def belief(t, z=None, u=None, dt=dt, R=R, Q=Q, S=20, D=40):
    if t == 0:
        x = np.array([0, 5]).reshape(2, 1)
        sigma = np.diag([0.01, 1])
        return x, sigma
    
    mu_pred, sigma_pred = predict(t, u, dt, R)
    H = partial_h_partial_x(mu_pred, dt, S, D)
    K = (sigma_pred @ H.T) @ inv(H @ sigma_pred @ H.T + Q)
    mu = mu_pred + K * (z - h(mu_pred, S, D))
    sigma = (np.eye(2) - K @ H) @ sigma_pred 
    return mu, sigma

mu, signma = belief(1, z1, u0)
print(mu)
print(signma)

[[2.51335109]
 [4.01854318]]
[[0.35841804 0.49780283]
 [0.49780283 1.09694837]]
